# 실습 04 · YOLO 학습 연결

> **Day15 | 객체탐지·YOLO·추적 · 01_기본필수**
> 원본 스크립트: `04_train.py` · 교안 PPT 53쪽 · 직접 구현: B05
> 결과: 이 노트북을 실제 실행한 출력

## 학습 목표

- `YOLO("yolo26n.yaml")`(구조)와 학습된 `.pt`의 차이를 확인한다.
- 1 epoch CPU 학습으로 runs 폴더·results.csv·best/last.pt가 생기는지 확인한다.
- 연결 성공과 탐지 품질을 구분한다.

## 핵심 개념

| 개념 | 설명 |
|---|---|
| `.yaml` | 층 구성 설정. 가중치는 무작위 |
| `.pt` | 학습된 가중치 + names·학습 인자 등 메타정보 |
| epoch | train 전체를 한 번 학습 |
| smoke test | 끝까지 돌아가는지만 보는 연결 점검 |

## 0. 준비

- 이 노트북은 `01_기본필수` 폴더(같은 폴더에 `data/`가 있는 위치)에서 연다.
- 입력은 `data/`의 원본을 읽기만 하고, 노트북이 만드는 결과 파일은 `outputs_notebook/`에 저장한다.
- CPU에서 실행한다. GPU·CUDA 코드는 없다.

In [1]:
# ============================================================
# 0. 라이브러리·경로 설정
# ============================================================
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# BASE : 노트북 폴더(01_기본필수) / DATA : 원본 입력 / OUT : 노트북 결과 폴더
BASE = Path.cwd()
DATA = BASE / "data"
OUT = BASE / "outputs_notebook"
OUT.mkdir(exist_ok=True)

def show(images, titles, size=3.2):
    """BGR 배열 여러 장을 한 줄로 표시한다.

    - images: BGR 배열 목록 (OpenCV로 읽은 그대로)
    - titles: 각 그림 제목 목록
    """
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size * 0.9))
    axes = np.atleast_1d(axes)
    for ax, img, title in zip(axes, images, titles):
        # cv2.cvtColor(BGR2RGB) : matplotlib은 RGB 순서로 그리므로 채널 순서를 바꿈
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 1. 데이터 설정 YAML

`train`·`val` 경로와 클래스 이름만 넣는다. test는 전달하지 않는다.

In [2]:
# ============================================================
# 1. 데이터 설정 YAML
# ============================================================
import yaml
import torch
import logging
from ultralytics import YOLO

# ultralytics 로그를 WARNING 이상만 출력 (학습 진행 표 생략)
logging.getLogger("ultralytics").setLevel(logging.WARNING)
torch.set_num_threads(1)
config = {"train": str(DATA/"images/train"), "val": str(DATA/"images/val"), "names": ["square", "circle", "triangle"]}
config_path = OUT / "p04_dataset.yaml"
config_path.write_text(yaml.safe_dump(config), encoding="utf-8")
print(config_path.read_text(encoding="utf-8"))

names:
- square
- circle
- triangle
train: "/home/user/work/out/01_\uAE30\uBCF8\uD544\uC218/data/images/train"
val: "/home/user/work/out/01_\uAE30\uBCF8\uD544\uC218/data/images/val"



## 2. 1 epoch 학습 (TODO B05)

In [3]:
# ============================================================
# 2. 1 epoch 학습 (TODO B05)
# ============================================================
model = YOLO("yolo26n.yaml")
# data=설정 YAML, epochs=1, imgsz=160, batch=4, device="cpu"
# workers=0, seed=42, project=outputs_notebook/p04_runs, name="smoke", exist_ok=True, verbose=False
model.train(data=str(config_path), epochs=1, imgsz=160, batch=4, device="cpu",
            workers=0, seed=42, project=str(OUT/"p04_runs"), name="smoke", exist_ok=True, verbose=False)
print("task:", model.task, "names:", model.names, "device:", model.device)

task: detect names: {0: 'square', 1: 'circle', 2: 'triangle'} device: cpu


## 3. 생성 파일과 학습 기록

In [4]:
# ============================================================
# 3. 생성 파일과 학습 기록
# ============================================================
run = OUT / "p04_runs/smoke"
print(sorted(p.name for p in run.iterdir()))
print(sorted(p.name for p in (run/"weights").iterdir()))
results = pd.read_csv(run / "results.csv")
results[["epoch", "train/box_loss", "train/cls_loss", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]]

['BoxF1_curve.png', 'BoxPR_curve.png', 'BoxP_curve.png', 'BoxR_curve.png', 'args.yaml', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'labels.jpg', 'results.csv', 'results.png', 'train_batch0.jpg', 'train_batch1.jpg', 'train_batch2.jpg', 'val_batch0_labels.jpg', 'val_batch0_pred.jpg', 'weights']
['best.pt', 'last.pt']


,epoch,train/box_loss,train/cls_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B)
0,1,3.08514,4.25777,0,0,0,0


## 4. `.yaml` 구조 vs `.pt` checkpoint

`.pt` 안에 무엇이 저장되는지 열어 본다. 직접 만든 파일만 `weights_only=False`로 연다.

In [5]:
# ============================================================
# 4. .yaml 구조 vs .pt checkpoint
# ============================================================
ckpt = torch.load(run / "weights/last.pt", map_location="cpu", weights_only=False)
print("checkpoint 키:", list(ckpt.keys()))
print("names:", ckpt["model"].names)
print("train_args 일부:", {k: ckpt["train_args"][k] for k in ["data", "epochs", "imgsz", "batch", "device"]})
print("파라미터 수:", sum(t.numel() for t in YOLO("yolo26n.yaml").model.parameters()))

checkpoint 키: ['date', 'version', 'license', 'docs', 'epoch', 'best_fitness', 'model', 'ema', 'updates', 'optimizer', 'scaler', 'train_args', 'train_metrics', 'train_results', 'git']
names: {0: 'square', 1: 'circle', 2: 'triangle'}
train_args 일부: {'data': '/home/user/work/out/01_기본필수/outputs_notebook/p04_dataset.yaml', 'epochs': 1, 'imgsz': 160, 'batch': 4, 'device': 'cpu'}
파라미터 수: 2572280


## 정리 (README 설명 질문)

- **1 epoch 연결 성공과 실제 탐지 품질이 다른 이유:** 위 표처럼 1 epoch 뒤 mAP는 0에 가깝다. 무작위 가중치에서 12장을 한 번 본 것으로는 박스를 배우지 못한다. 이 실습이 확인한 것은 데이터 경로·라벨·학습 루프·저장이 끝까지 연결된다는 사실이다.
- **`.yaml`과 `.pt`의 차이:** `.yaml`은 층 구성만 있어 매번 무작위로 시작한다. `.pt`에는 학습된 가중치, 클래스 이름, 학습 인자, 날짜·버전 같은 메타정보가 들어 있다.